# Laboratorio 5 – Modelo espacial de cobertura de servicios de salud
**CC2017 Modelación y Simulación** · Territorio de análisis: **Guatemala** (22 departamentos = nivel 1, 354 municipios = nivel 2)

Este notebook cubre el **Task 1** y el **Task 2**. Las fuentes son GADM 4.1 (divisiones administrativas), Global Healthsites Mapping Project (instalaciones) y WorldPop 2020 a 1 km (población).

> **Estado de ejecución:** los datos de Healthsites requieren una API key aprobada por los administradores. Las celdas que dependen de esos datos (desde 1.1c) se ejecutan en cuanto exista `data/healthsites_gtm.geojson` o la llave quede activa en `.env` (`HEALTHSITES_API_KEY=...`).

### Convenciones del código
* Todas las distancias se calculan en **EPSG:32615** (metros).
* La población de cada celda de WorldPop se asigna a un municipio por la posición de su **centro** (equivale a `rasterio.mask` con `all_touched=False`; se verifica en 1.3b).
* Las unidades de WorldPop son **personas por celda** (conteo), no densidad.

## Task 1.1 – Instalación y descarga de datos

In [1]:
# Instalación (solo la primera vez):
# %pip install geopandas pandas numpy matplotlib requests shapely rasterio scipy pyproj pyogrio matplotlib-scalebar python-dotenv

import os, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import shapely
import requests
import rasterio
import rasterio.transform
from rasterio.mask import mask as rio_mask
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib_scalebar.scalebar import ScaleBar
from scipy.spatial import cKDTree
from pyproj import Proj
from dotenv import load_dotenv

warnings.filterwarnings("ignore", category=UserWarning)
DATA = Path("data"); DATA.mkdir(exist_ok=True)
FIG = Path("figures"); FIG.mkdir(exist_ok=True)
ISO3, COUNTRY = "GTM", "Guatemala"
UTM = "EPSG:32615"          # CRS proyectado (justificado en 1.2a)
RADII = [5, 10, 20]         # radios de buffer en km

def download(url, dest):
    """Descarga reproducible: no vuelve a bajar el archivo si ya existe."""
    dest = Path(dest)
    if not dest.exists():
        r = requests.get(url, timeout=300); r.raise_for_status()
        dest.write_bytes(r.content)
    return dest

# 1.1a – versiones
print("geopandas :", gpd.__version__)
print("shapely   :", shapely.__version__)
print("pandas    :", pd.__version__, "| numpy:", np.__version__, "| rasterio:", rasterio.__version__)

geopandas : 1.2.0
shapely   : 2.1.2
pandas    : 2.3.3 | numpy: 2.4.2 | rasterio: 1.5.1


### 1.1b – Polígonos administrativos (GADM 4.1)
La página `gadm.org/download_country.html` del enunciado es un formulario web, no un archivo. GADM sirve los GeoJSON por URL directa con el patrón
`https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_<ISO3>_<nivel>.json`.

In [2]:
GADM = "https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_{iso}_{lvl}.json"
gadm = {}
for lvl in (1, 2):
    f = download(GADM.format(iso=ISO3, lvl=lvl), DATA / f"gadm41_{ISO3}_{lvl}.json")
    gadm[lvl] = gpd.read_file(f)
    assert gadm[lvl].crs.to_epsg() == 4326, "El CRS debe ser WGS84"
gadm1, gadm2 = gadm[1], gadm[2]
print(f"Nivel 1: {len(gadm1)} divisiones | CRS {gadm1.crs}")
print(f"Nivel 2: {len(gadm2)} divisiones | CRS {gadm2.crs}")
print("Tipos nivel 2:", gadm2["ENGTYPE_2"].value_counts().to_dict())

Nivel 1: 22 divisiones | CRS EPSG:4326
Nivel 2: 354 divisiones | CRS EPSG:4326
Tipos nivel 2: {'Municipality': 352, 'Waterbody': 2}


### 1.1c – Instalaciones de salud (Global Healthsites Mapping Project)
* El endpoint `/api/v2/facilities/` del enunciado está **deprecado** (responde: *"This API is deprecated, please upgrade your application to use v3"*). Se usa **`/api/v3/facilities/`**.
* La API v3 es paginada y filtra por `country` o `extent`, pero no por polígono; por eso se descarga por país y luego se aplica la intersección espacial con el nivel 1 de GADM.
* La llave se lee de `.env` (no se sube al repositorio).
* El resultado se guarda en `data/healthsites_gtm.geojson`; si ya existe, no se vuelve a consultar la API.
* La documentación pública no detalla el esquema de la respuesta. El parser acepta `FeatureCollection` o lista de *features*, y toma el tipo de instalación de `amenity` o `healthcare`. **Revisar al primer uso con datos reales.**

In [3]:
load_dotenv(".env")
API_KEY = os.getenv("HEALTHSITES_API_KEY")
HS_URL = "https://healthsites.io/api/v3/facilities/"
HS_CACHE = DATA / "healthsites_gtm.geojson"

def fetch_healthsites(api_key, country=COUNTRY, max_pages=1000, pause=0.3):
    """Descarga todas las páginas de la API v3 como lista de features GeoJSON."""
    feats = []
    for page in range(1, max_pages + 1):
        r = requests.get(HS_URL, timeout=120, params={
            "api-key": api_key, "country": country, "page": page,
            "output": "geojson", "flat-properties": "true"})
        if r.status_code == 403:
            raise PermissionError(r.json().get("detail", r.text))
        if r.status_code == 404:          # página fuera de rango = fin
            break
        r.raise_for_status()
        payload = r.json()
        batch = payload.get("features", []) if isinstance(payload, dict) else payload
        if not batch:
            break
        feats.extend(batch)
        time.sleep(pause)
    return feats

fac_raw = None
if HS_CACHE.exists():
    fac_raw = gpd.read_file(HS_CACHE)
else:
    try:
        feats = fetch_healthsites(API_KEY)
        fac_raw = gpd.GeoDataFrame.from_features(feats, crs="EPSG:4326")
        fac_raw.to_file(HS_CACHE, driver="GeoJSON")
    except PermissionError as e:
        print("Healthsites no disponible todavía:", e)

fac = None
if fac_raw is not None:
    # Normalizar geometría a puntos y tipo de instalación
    fac_raw["geometry"] = fac_raw.geometry.representative_point()
    tipo = pd.Series(np.nan, index=fac_raw.index, dtype=object)
    for col in ("amenity", "healthcare"):
        if col in fac_raw.columns:
            tipo = tipo.fillna(fac_raw[col])
    fac_raw["facility_type"] = tipo.fillna("unknown")
    # Intersección espacial con el nivel 1 de GADM (se queda solo con lo que está en Guatemala)
    fac = gpd.sjoin(fac_raw, gadm1[["NAME_1", "geometry"]], predicate="intersects", how="inner")
    fac = fac[~fac.index.duplicated()].drop(columns="index_right").reset_index(drop=True)
    print(f"Instalaciones descargadas: {len(fac_raw)} | dentro de Guatemala: {len(fac)}")
    print(fac["facility_type"].value_counts().to_string())

Healthsites no disponible todavía: This API key is not active. Please wait for it to be approved by the admins.


### 1.1d – Ráster de población (WorldPop 2020, 1 km)

In [4]:
# Dataset "Global_2000_2020_1km" (último año con datos observados: 2020). La serie R2025A llega a 2030 pero es una proyección en versión alfa.
WP = f"https://data.worldpop.org/GIS/Population/Global_2000_2020_1km/2020/{ISO3}/{ISO3.lower()}_ppp_2020_1km_Aggregated.tif"
RASTER = download(WP, DATA / f"{ISO3.lower()}_ppp_2020_1km_Aggregated.tif")

with rasterio.open(RASTER) as src:
    pop_arr = src.read(1, masked=True)
    pop_tf, pop_crs, nrows, ncols = src.transform, src.crs, src.height, src.width
    res_deg = src.res

lat0 = gadm1.total_bounds[[1, 3]].mean()
res_m = (res_deg[0] * 111_320 * np.cos(np.radians(lat0)), res_deg[1] * 110_574)
print(f"CRS                : {pop_crs}")
print(f"Resolución         : {res_deg[0]:.6f}° ≈ {res_m[0]:.0f} m (E-O) × {res_m[1]:.0f} m (N-S) a lat {lat0:.1f}°")
print(f"Filas × columnas   : {nrows} × {ncols}")
print(f"Máximo por celda   : {pop_arr.max():,.1f} personas/celda  (≈ {pop_arr.max() / (res_m[0]*res_m[1]/1e6):,.0f} hab/km²)")
print(f"Población total    : {pop_arr.sum():,.0f}")

CRS                : EPSG:4326
Resolución         : 0.008333° ≈ 893 m (E-O) × 921 m (N-S) a lat 15.8°
Filas × columnas   : 490 × 479
Máximo por celda   : 20,086.5 personas/celda  (≈ 24,419 hab/km²)
Población total    : 18,265,862


## Task 1.2 – Análisis exploratorio espacial

### 1.2a – Reproyección

In [ ]:
assert fac is not None, "Faltan datos de Healthsites: espere la aprobación de la API key (o coloque data/healthsites_gtm.geojson)."
gadm1_utm, gadm2_utm, fac_utm = (g.to_crs(UTM) for g in (gadm1, gadm2, fac))

# Tabla de píxeles con población: el centro de cada celda hereda su municipio (nivel 2)
rows, cols = np.where(~np.ma.getmaskarray(pop_arr) & (pop_arr.data > 0))
xs, ys = rasterio.transform.xy(pop_tf, rows, cols)
pix = gpd.GeoDataFrame({"pop": pop_arr.data[rows, cols].astype(float)},
                       geometry=gpd.points_from_xy(xs, ys), crs="EPSG:4326")
pix = gpd.sjoin(pix, gadm2[["GID_2", "GID_1", "geometry"]], predicate="within", how="inner")
pix = pix[~pix.index.duplicated()].drop(columns="index_right").to_crs(UTM)
pix = pd.DataFrame({"pop": pix["pop"], "GID_1": pix["GID_1"], "GID_2": pix["GID_2"],
                    "px": pix.geometry.x, "py": pix.geometry.y}).reset_index(drop=True)
px, py = pix["px"].values, pix["py"].values
POP_TOTAL = pix["pop"].sum()
print(f"Celdas pobladas: {len(pix):,} | población asignada a municipios: {POP_TOTAL:,.0f} de {pop_arr.sum():,.0f} en el ráster")

# Distorsión del CRS elegido: factor de escala k en puntos extremos del país
p = Proj(UTM)
b = gadm1.total_bounds
print("\nFactor de escala k de EPSG:32615 (k=1 sin distorsión):")
for name, lon, lat in [("Oeste (San Marcos)", b[0], 15.0), ("Centro (Guatemala)", -90.5, 14.6),
                       ("Este (Izabal)", b[2], 15.7), ("Norte (Petén)", -90.0, b[3]), ("Sur", -91.0, b[1])]:
    print(f"  {name:20s} lon {lon:7.2f}  k = {p.get_factors(lon, lat).meridional_scale:.5f}")

**Justificación del CRS: EPSG:32615 (WGS 84 / UTM zona 15N).**
* La proyección **UTM es transversa de Mercator, conforme (conserva ángulos y formas locales)**, con factor de escala `k = 0.9996` en el meridiano central (93° O). Las distancias en metros se obtienen directamente de las coordenadas.
* Guatemala (≈ 88.2° O a 92.3° O) queda casi toda en la zona 15 (96° O a 90° O); el extremo oriental pasa a la zona 16. La tabla anterior muestra que incluso en el punto más lejano del meridiano central `k` apenas se aparta de 1 en ≈ 0.3 %, es decir **≈ 300 m de error por cada 100 km**. Esa distorsión es despreciable frente al radio más grande (20 km → error ≈ 60 m) y frente a la resolución del ráster (≈ 900 m).
* Se prefirió una sola zona frente a EPSG:32616 porque la mayor parte del territorio y de la población está en la zona 15; usar una zona única evita partir el país y mantiene un único sistema para buffers y distancias.
* Alternativas: *Guatemala Norte/Sur* (EPSG:5559/5459) parten el país en dos CRS, y `ESRI:103598` (GTM) tendría menor distorsión pero no es un código EPSG.

### 1.2b – Mapa base (tres capas)

In [ ]:
def north_arrow(ax, x=0.95, y=0.93):
    ax.annotate("N", xy=(x, y), xytext=(x, y - 0.09), xycoords="axes fraction", ha="center", fontsize=13,
                arrowprops=dict(arrowstyle="-|>", color="k", lw=2))

types = fac_utm["facility_type"].value_counts()
main_types = list(types.index[:7])
palette = dict(zip(main_types, plt.cm.tab10.colors))
fig, ax = plt.subplots(figsize=(9, 9))
gadm2_utm.plot(ax=ax, color="#d9d9d9", edgecolor="white", linewidth=0.2)
for t in main_types:
    s = fac_utm[fac_utm["facility_type"] == t]
    ax.scatter(s.geometry.x, s.geometry.y, s=9, color=palette[t], label=f"{t} ({len(s)})", zorder=3)
other = fac_utm[~fac_utm["facility_type"].isin(main_types)]
if len(other):
    ax.scatter(other.geometry.x, other.geometry.y, s=9, color="k", label=f"otros ({len(other)})", zorder=3)
gadm1_utm.boundary.plot(ax=ax, color="#222", linewidth=0.9)
ax.add_artist(ScaleBar(1, "m", length_fraction=0.25, location="lower left"))
north_arrow(ax)
ax.legend(title="Tipo de instalación", loc="lower right", fontsize=8)
ax.set_title("Línea base: instalaciones de salud en Guatemala (EPSG:32615)")
ax.set_axis_off(); plt.tight_layout(); plt.savefig(FIG / "mapa_base.png", dpi=150); plt.show()

### 1.2c – Estadísticas por departamento (nivel 1)

In [ ]:
dep = gadm1_utm[["GID_1", "NAME_1", "geometry"]].set_index("GID_1")
dep["area_km2"] = dep.geometry.area / 1e6
dep["poblacion"] = pix.groupby("GID_1")["pop"].sum()

fac_dep = gpd.sjoin(fac_utm[["geometry"]], gadm1_utm[["GID_1", "geometry"]], predicate="within", how="left")
fac_dep = fac_dep[~fac_dep.index.duplicated()]
fac_utm["GID_1"] = fac_dep["GID_1"].values
dep["n_instalaciones"] = fac_utm.groupby("GID_1").size().reindex(dep.index).fillna(0).astype(int)
dep["inst_x_100k"] = dep["n_instalaciones"] / dep["poblacion"] * 1e5   # densidad por 100,000 hab.

def mean_nn(g):
    """Distancia media al vecino más cercano entre instalaciones del mismo departamento (m)."""
    if len(g) < 2: return np.nan
    xy = np.c_[g.geometry.x, g.geometry.y]
    return cKDTree(xy).query(xy, k=2)[0][:, 1].mean()
dep["dist_nn_km"] = pd.Series({k: mean_nn(g) for k, g in fac_utm.groupby("GID_1")}).reindex(dep.index) / 1000

tabla_dep = dep.drop(columns="geometry").sort_values("inst_x_100k")
display(tabla_dep.round(2))
print("Tres departamentos con menor densidad de cobertura (instalaciones por 100,000 hab.):")
print(tabla_dep.head(3)[["NAME_1", "n_instalaciones", "inst_x_100k"]].round(2).to_string())

## Task 1.3 – Buffers de cobertura

### 1.3a – Buffers y unión
Para cada radio *r* se construye `C_r = ⋃_i B(f_i, r)`, donde `B(f_i, r)` es el disco de radio *r* centrado en la instalación *f_i*. En el código, `unary_union` implementa esa unión.

In [ ]:
fx, fy = fac_utm.geometry.x.values, fac_utm.geometry.y.values
coverage = {r: fac_utm.buffer(r * 1000).union_all() for r in RADII}
for r in RADII:
    print(f"Radio {r:>2} km: área de cobertura unificada = {coverage[r].area / 1e6:,.0f} km²")

### 1.3b – Población cubierta
`Pob_cubierta(r) = Σ_{celdas c ∈ C_r} pop_c`, y el porcentaje es `Pob_cubierta / Pob_total`.

In [ ]:
cov_flag = {r: shapely.contains_xy(coverage[r], px, py) for r in RADII}   # celdas dentro de la cobertura
filas = []
for r in RADII:
    cub = pix.loc[cov_flag[r], "pop"].sum()
    filas.append({"radio_km": r, "poblacion_cubierta": cub, "poblacion_total": POP_TOTAL,
                  "cobertura_%": 100 * cub / POP_TOTAL})
tabla_cob = pd.DataFrame(filas)
display(tabla_cob.round(2))

# Verificación independiente con rasterio.mask (centros de celda dentro del polígono)
geom4326 = gpd.GeoSeries([coverage[10]], crs=UTM).to_crs(4326)
with rasterio.open(RASTER) as src:
    masked, _ = rio_mask(src, geom4326, crop=False, all_touched=False, filled=False)
sum_mask = float(masked[0].sum())
print(f"Verificación r = 10 km → rasterio.mask: {sum_mask:,.0f} | tabla de celdas: {pix.loc[cov_flag[10], 'pop'].sum():,.0f}")
print("La pequeña diferencia (~0.2 %) viene de celdas costeras/fronterizas cuyo centro queda fuera de los polígonos GADM, "
      "que la tabla de celdas excluye de la población total.")

### 1.3c – Mapa de cobertura a 10 km por municipio

In [ ]:
cov10 = coverage[10]
inter_area = gadm2_utm.geometry.intersection(cov10).area
frac = (inter_area / gadm2_utm.geometry.area).values
gadm2_utm["frac_cubierta"] = frac
gadm2_utm["estado"] = np.select([frac >= 0.99, frac > 0], ["Cubierto", "Parcialmente cubierto"], "Sin cobertura")
print(gadm2_utm["estado"].value_counts().to_string())

orden = ["Sin cobertura", "Parcialmente cubierto", "Cubierto"]
cmap = ListedColormap(["#b2182b", "#f4e285", "#2166ac"])           # divergente: rojo (sin cobertura) – amarillo – azul
fig, ax = plt.subplots(figsize=(9, 9))
gadm2_utm["estado"] = pd.Categorical(gadm2_utm["estado"], orden)
gadm2_utm.plot(column="estado", cmap=cmap, categorical=True, legend=True, ax=ax, edgecolor="white", linewidth=0.2,
               legend_kwds={"loc": "lower right", "title": "Cobertura a 10 km"})
gadm1_utm.boundary.plot(ax=ax, color="#333", linewidth=0.6)
ax.scatter(fx, fy, s=6, color="k", zorder=3, label="Instalaciones")
ax.add_artist(ScaleBar(1, "m", length_fraction=0.25, location="lower left")); north_arrow(ax)
ax.set_title("Cobertura de salud a 10 km por municipio (área cubierta ≥ 99 % = cubierto)")
ax.set_axis_off(); plt.tight_layout(); plt.savefig(FIG / "cobertura_10km.png", dpi=150); plt.show()

## Task 2.1 – Sensibilidad de la cobertura

Se calcula, para cada celda poblada, la distancia a la instalación más cercana: `d_c = min_i ‖c − f_i‖`. Con ella, el porcentaje cubierto a radio *r* es `Σ_c pop_c · 1[d_c ≤ r] / Σ_c pop_c`, equivalente a la unión de buffers del Task 1.3.

In [ ]:
tree_fac = cKDTree(np.c_[fx, fy])
pix["d_fac"], pix["i_fac"] = tree_fac.query(np.c_[px, py])        # distancia (m) e índice de la instalación más cercana
assert np.isclose(pix.loc[pix.d_fac <= 10_000, "pop"].sum(), pix.loc[cov_flag[10], "pop"].sum(), rtol=1e-3)
pix["unc10"] = np.where(pix["d_fac"] > 10_000, pix["pop"], 0.0)   # población sin cobertura a 10 km
nombres = gadm2.set_index("GID_2")[["NAME_2", "NAME_1"]]

### 2.1a y 2.1b – Municipios con mayor población sin cobertura (10 km)
* **Distancia a la instalación más cercana:** desde el centroide ponderado por la población *no cubierta* del municipio.
* **Radio mínimo de cobertura total:** distancia máxima de una celda poblada del municipio a su instalación más cercana. Con un buffer de ese radio ninguna celda poblada queda fuera.
* **Tiempo de viaje:** `t = d / 40 km/h`. Es un **límite inferior**, porque la distancia es en línea recta y no sigue la red vial.

In [ ]:
top5 = pix.groupby("GID_2")["unc10"].sum().sort_values(ascending=False).head(5)
filas = []
for gid, unc in top5.items():
    sub = pix[pix.GID_2 == gid]
    u = sub[sub.unc10 > 0]
    cx, cy = np.average(u.px, weights=u.unc10), np.average(u.py, weights=u.unc10)
    d, j = tree_fac.query([cx, cy])
    r_min = sub.d_fac.max() / 1000
    filas.append({"municipio": nombres.loc[gid, "NAME_2"], "departamento": nombres.loc[gid, "NAME_1"],
                  "pob_sin_cobertura": unc, "dist_instalacion_km": d / 1000,
                  "tipo_mas_cercana": fac_utm.loc[j, "facility_type"],
                  "radio_min_km": r_min, "tiempo_min_a_40kmh": r_min / 40 * 60})
tabla_top5 = pd.DataFrame(filas)
display(tabla_top5.round(1))

**Interpretación 2.1b** *(completar con los resultados de la tabla anterior)*: indicar, para cada municipio, qué implica el radio mínimo en minutos de viaje, y recordar que sobre carreteras reales (sinuosidad, caminos de terracería) el tiempo real será mayor que `d/40`.

### 2.1c – Curva de cobertura acumulada (1–50 km)

In [ ]:
radios = np.arange(1, 51)
cobertura = np.array([pix.loc[pix.d_fac <= r * 1000, "pop"].sum() / POP_TOTAL * 100 for r in radios])

# Punto de inflexión por el método del codo (Kneedle): máxima separación entre la curva normalizada y la diagonal
xn = (radios - radios.min()) / (radios.max() - radios.min())
yn = (cobertura - cobertura.min()) / (cobertura.max() - cobertura.min())
r_knee = radios[np.argmax(yn - xn)]
ganancia = np.diff(cobertura, prepend=0)    # puntos porcentuales ganados por cada km adicional

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(radios, cobertura, lw=2, color="#2166ac")
ax.axvline(r_knee, ls="--", color="#b2182b"); ax.scatter([r_knee], [cobertura[r_knee - 1]], color="#b2182b", zorder=3)
ax.annotate(f"codo: {r_knee} km\n{cobertura[r_knee-1]:.1f} %", (r_knee, cobertura[r_knee - 1]),
            xytext=(r_knee + 3, cobertura[r_knee - 1] - 12))
ax.set_xlabel("Radio de buffer (km)"); ax.set_ylabel("Población cubierta (%)"); ax.grid(alpha=0.3)
ax.set_title("Curva de cobertura acumulada"); plt.tight_layout(); plt.savefig(FIG / "curva_cobertura.png", dpi=150); plt.show()
print(f"Punto de inflexión (codo): {r_knee} km → {cobertura[r_knee-1]:.1f} % de cobertura")
print(f"Ganancia marginal antes del codo ≈ {ganancia[:r_knee].mean():.2f} pp/km; después ≈ {ganancia[r_knee:].mean():.2f} pp/km")

**Interpretación 2.1c** *(completar con el codo obtenido)*: pasado el punto de inflexión, cada kilómetro extra de radio agrega muy poca población; la política de acceso deja de resolverse con "más cobertura" y requiere intervenir en las zonas remotas restantes con otros medios (unidades móviles, telemedicina, transporte) y no con instalaciones fijas de mayor alcance.

## Task 2.2 – Índice compuesto de vulnerabilidad (nivel 2)

Para el municipio *m*:
* **C1** = distancia media a la instalación más cercana, **ponderada por población** (más lejos → más vulnerable).
* **C2** = población sin cobertura a 10 km / área en km² (mayor → más vulnerable).
* **C3** = instalaciones por 10,000 habitantes (menor → más vulnerable; se invierte tras normalizar).

In [ ]:
muni = gadm2_utm[["GID_2", "NAME_2", "NAME_1", "geometry"]].set_index("GID_2")
muni["area_km2"] = muni.geometry.area / 1e6
g = pix.assign(w_d=pix["d_fac"] * pix["pop"]).groupby("GID_2")
muni["poblacion"] = g["pop"].sum()
muni["unc10"] = g["unc10"].sum()
muni["c1_raw"] = g["w_d"].sum() / muni["poblacion"] / 1000                      # km
muni["c2_raw"] = muni["unc10"] / muni["area_km2"]                               # hab/km²
fm = gpd.sjoin(fac_utm[["geometry"]], muni.reset_index()[["GID_2", "geometry"]], predicate="within", how="left")
muni["n_inst"] = fm[~fm.index.duplicated()].groupby("GID_2").size().reindex(muni.index).fillna(0)
muni["c3_raw"] = muni["n_inst"] / muni["poblacion"] * 1e4                       # inst. por 10,000 hab.

valid = muni["poblacion"].fillna(0) > 0
print(f"Municipios sin población en el ráster (excluidos del índice): {(~valid).sum()}")

minmax = lambda s: (s - s.min()) / (s.max() - s.min())
muni.loc[valid, "c1"] = minmax(muni.loc[valid, "c1_raw"])
muni.loc[valid, "c2"] = minmax(muni.loc[valid, "c2_raw"])
muni.loc[valid, "c3"] = 1 - minmax(muni.loc[valid, "c3_raw"])                   # invertido: pocas instalaciones → 1
display(muni.loc[valid, ["c1_raw", "c2_raw", "c3_raw", "c1", "c2", "c3"]].describe().round(3))

### 2.2a – Justificación formal de la normalización min-max
`x' = (x − x_min) / (x_max − x_min)` es una transformación afín creciente, de modo que:
1. **Conserva el orden y las proporciones relativas** entre municipios (no altera el ranking dentro de cada componente).
2. **Lleva las tres variables a [0, 1] sin unidades** (km, hab/km², instalaciones/10,000 hab.), lo que permite promediarlas.
3. Es **interpretable**: 0 = el municipio menos vulnerable en ese componente y 1 = el más vulnerable.

**Cuándo puede no ser apropiada:** (i) es **sensible a valores atípicos**: un solo municipio con densidad extrema (por ejemplo, un municipio pequeño con muchas instalaciones en C3, o la capital en C2) comprime al resto cerca de 0; en ese caso conviene *winsorizar*, usar una transformación logarítmica o *z-scores* robustos; (ii) los extremos dependen de la muestra, así que los índices no son comparables entre territorios o años; (iii) supone que la relación entre el componente y la vulnerabilidad es **lineal**, cuando la vulnerabilidad real puede tener umbrales (por ejemplo, 60 minutos de viaje).

### 2.2b – Pesos del índice

| Componente | Peso | Argumento |
|---|---|---|
| C1 – distancia a la instalación | **0.40** | La distancia es la barrera de acceso más directa y la que se puede intervenir con infraestructura. En emergencias (partos, trauma) el tiempo de llegada define el resultado clínico. |
| C2 – población sin cobertura por km² | **0.35** | Mide cuánta gente concentra la brecha: una brecha en una zona densa afecta a más personas por unidad de territorio y rinde más por cada nueva instalación. |
| C3 – instalaciones por 10,000 hab. | **0.25** | Refleja la capacidad local, pero es un indicador más ruidoso: cuenta instalaciones sin distinguir tamaño ni calidad (una farmacia pesa igual que un hospital), y Healthsites tiene subregistro de instalaciones. |

Los pesos suman 1. El análisis de sensibilidad a otros pesos corresponde al Task 3.3.

In [ ]:
WEIGHTS = {"c1": 0.40, "c2": 0.35, "c3": 0.25}
assert abs(sum(WEIGHTS.values()) - 1) < 1e-9
muni["indice"] = sum(w * muni[c] for c, w in WEIGHTS.items())
top10 = muni.dropna(subset=["indice"]).sort_values("indice", ascending=False).head(10)
display(top10[["NAME_2", "NAME_1", "poblacion", "c1_raw", "c2_raw", "c3_raw", "indice"]].round(3))

### 2.2c – Mapa coroplético por quintiles

In [ ]:
labels = ["Muy baja", "Baja", "Media", "Alta", "Muy alta"]
muni["categoria"] = pd.qcut(muni["indice"].rank(method="first"), 5, labels=labels)
fig, ax = plt.subplots(figsize=(10, 10))
muni.plot(ax=ax, color="#eeeeee", edgecolor="white", linewidth=0.2)             # municipios sin índice
muni.dropna(subset=["categoria"]).plot(column="categoria", cmap="YlOrRd", categorical=True, legend=True, ax=ax,
                                       edgecolor="white", linewidth=0.2,
                                       legend_kwds={"loc": "lower right", "title": "Vulnerabilidad"})
gadm1_utm.boundary.plot(ax=ax, color="#333", linewidth=0.6)
for k, (gid, r) in enumerate(top10.iterrows(), 1):
    pt = r.geometry.representative_point()
    ax.annotate(f"{k}. {r['NAME_2']}", (pt.x, pt.y), fontsize=7, ha="center", zorder=5,
                bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.8))
ax.add_artist(ScaleBar(1, "m", length_fraction=0.25, location="lower left")); north_arrow(ax)
ax.set_title("Índice de vulnerabilidad espacial por municipio (quintiles) – top 10 etiquetado")
ax.set_axis_off(); plt.tight_layout(); plt.savefig(FIG / "indice_vulnerabilidad.png", dpi=150); plt.show()

## Task 2.3 – MCLP simplificado (algoritmo voraz)

Se busca maximizar `Σ_c pop_c · y_c`, donde `y_c = 1` si la celda *c* queda cubierta, eligiendo *p = 5* sitios de entre los candidatos. El greedy elige, en cada iteración, el sitio con mayor ganancia marginal sobre la población aún no cubierta.

### 2.3a – Grilla de candidatos

In [ ]:
GRID = 20_000
minx, miny, maxx, maxy = gadm1_utm.total_bounds
gx, gy = np.meshgrid(np.arange(minx, maxx, GRID), np.arange(miny, maxy, GRID))
land = gadm1_utm.union_all()
water = gadm2_utm[gadm2_utm["ENGTYPE_2"].fillna("").str.contains("water", case=False)]
if len(water):
    land = land.difference(water.union_all())          # excluye cuerpos de agua marcados en GADM
inside = shapely.contains_xy(land, gx.ravel(), gy.ravel())
cand = np.c_[gx.ravel()[inside], gy.ravel()[inside]]
print(f"Puntos de la grilla: {gx.size} | candidatos dentro de Guatemala (sin agua): {len(cand)} | polígonos de agua en GADM: {len(water)}")

### 2.3b – Población adicional por candidato (radio 10 km)
Ganancia = población de las celdas **sin cobertura actual** que caen a ≤ 10 km del candidato.

In [ ]:
R = 10_000
unc_idx = np.where(pix["d_fac"].values > R)[0]
popu = pix["pop"].values[unc_idx]
tree_unc = cKDTree(np.c_[px[unc_idx], py[unc_idx]])
neigh = [np.array(n, dtype=int) for n in tree_unc.query_ball_point(cand, r=R)]
gain0 = np.array([popu[n].sum() for n in neigh])
print(f"Ganancia individual: máx {gain0.max():,.0f} | mediana {np.median(gain0):,.0f} | candidatos con ganancia 0: {(gain0 == 0).sum()}")

### 2.3c – Selección voraz de 5 sitios

In [ ]:
P = 5
remaining = np.ones(len(popu), dtype=bool)         # celdas aún sin cobertura
chosen, filas = [], []
cubierta_base = pix.loc[pix["d_fac"] <= R, "pop"].sum()
acum = 0.0
for it in range(1, P + 1):
    gains = np.array([popu[n][remaining[n]].sum() if k not in chosen else -1 for k, n in enumerate(neigh)])
    k = int(np.argmax(gains))
    chosen.append(k)
    remaining[neigh[k]] = False                    # actualiza el mapa de cobertura
    acum += gains[k]
    filas.append({"iteracion": it, "x_utm": cand[k, 0], "y_utm": cand[k, 1], "pob_adicional": gains[k],
                  "pob_adicional_acum": acum, "cobertura_total_%": 100 * (cubierta_base + acum) / POP_TOTAL})
res_mclp = pd.DataFrame(filas)
display(res_mclp.round(1))
print(f"Cobertura a 10 km antes: {100*cubierta_base/POP_TOTAL:.2f} % → después de 5 sitios: {res_mclp['cobertura_total_%'].iloc[-1]:.2f} %")

sel = gpd.GeoDataFrame(geometry=gpd.points_from_xy(cand[chosen, 0], cand[chosen, 1]), crs=UTM)
nuevo = sel.buffer(R)
fig, ax = plt.subplots(figsize=(9, 9))
gadm2_utm.plot(ax=ax, color="#e6e6e6", edgecolor="white", linewidth=0.2)
gpd.GeoSeries([coverage[10]], crs=UTM).plot(ax=ax, color="#9ecae1", alpha=0.6)
nuevo.plot(ax=ax, color="#fd8d3c", alpha=0.6)
gadm1_utm.boundary.plot(ax=ax, color="#333", linewidth=0.6)
ax.scatter(fx, fy, s=5, color="k", zorder=3, label="Instalaciones existentes")
sel.plot(ax=ax, color="#b2182b", markersize=60, marker="*", zorder=4, label="Sitios propuestos")
for k, (x, y) in enumerate(cand[chosen], 1):
    ax.annotate(str(k), (x, y), xytext=(5, 5), textcoords="offset points", fontsize=11, weight="bold")
ax.legend(loc="lower right"); ax.add_artist(ScaleBar(1, "m", length_fraction=0.25, location="lower left")); north_arrow(ax)
ax.set_title("MCLP voraz: 5 nuevos sitios (azul: cobertura actual 10 km; naranja: cobertura nueva)")
ax.set_axis_off(); plt.tight_layout(); plt.savefig(FIG / "mclp_voraz.png", dpi=150); plt.show()

## Uso de IA generativa
> Nota del enunciado: indicar el prompt utilizado por task y por qué funcionó.

* **Prompt utilizado:** *(completar)*
* **Por qué funcionó:** *(completar)*